# Section 1: HORIZON = 2 Prediction System (10-Minute Direction)

## Cell 1: Horizon Configuration & Purged CV Folds
- **Function:** Initializes project environment, locks horizon parameter $N=2$ (10 minutes, 2 candles forward), and verifies 6-fold expanding purged walk-forward cross-validation schedule.
- **Runtime:** fast (~2s)
- **Status:** Configuration check (no training)

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Set project root and python path
PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT / "src").exists():
    ML_DIR = PROJECT_ROOT
elif (PROJECT_ROOT / "ML" / "src").exists():
    ML_DIR = PROJECT_ROOT / "ML"
else:
    ML_DIR = PROJECT_ROOT.parent

if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

from src import config
from src.cv import PurgedWalkForwardCV
from src.column_dictionary import load_master

# Set target horizon
HORIZON = 2
MINUTES = HORIZON * config.INTERVAL_MINUTES

print("=" * 70)
print(f"HORIZON CONFIGURATION: N = {HORIZON} ({MINUTES} Minutes / 2 Candles)")
print("=" * 70)
print(f"Working Directory      : {ML_DIR}")
print(f"Binary Target          : {config.BINARY_TARGET}")
print(f"Three-Class Target     : {config.THREE_CLASS_TARGET}")
print(f"Directional Benchmark  : {config.TRUTH_COL}")
print(f"Forward Return Column  : {config.RETURN_COL}")
print(f"Target Validity Mask   : {config.VALID_COL}")
print(f"Round-Trip Friction    : {config.COST * 10000:.1f} bps ({config.COST})")
print(f"Training Stride        : Every {config.STRIDE}-rd row (stride downsampling)")
print(f"OpenMP / CPU Cores     : n_jobs = {config.N_JOBS}")
print(f"Random Seed            : {config.SEED}")

# Cross-Validation Folds Verification
cv = PurgedWalkForwardCV()
df_train = load_master(split="train")
print(f"\nLoaded TRAIN split: {len(df_train):,} total rows (Warmup offset: {cv.warmup_candles:,})")

fold_info = []
for fold, tr_idx, te_idx in cv.split(df_train):
    tr_start, tr_end = df_train.index[tr_idx[0]], df_train.index[tr_idx[-1]]
    te_start, te_end = df_train.index[te_idx[0]], df_train.index[te_idx[-1]]
    purge_gap = te_idx[0] - tr_idx[-1] - 1
    fold_info.append({
        "Fold": fold,
        "Train Start": str(tr_start)[:10],
        "Train End": str(tr_end)[:10],
        "Purge Rows": purge_gap,
        "Test Start": str(te_start)[:10],
        "Test End": str(te_end)[:10],
        "Test Rows": len(te_idx),
    })

df_cv_summary = pd.DataFrame(fold_info)
display(df_cv_summary)
assert all(r["Purge Rows"] == 12 for r in fold_info), "Purge gap must be exactly 12 candles!"
print("Cross-Validation fold boundaries verified: exactly 12-candle purge across all 6 folds.")

## Cell 2: Data Preparation & Column Integrity Audit
- **Function:** Inspects class balance of `bin_dir_2` and `target_2_S1` on TRAIN, prints feature counts and names for linear (70) and non-linear (136) models, and asserts that no input feature possesses a forbidden role.
- **Runtime:** fast (~15s)
- **Status:** Integrity audit (no training)

In [ ]:
from src.column_dictionary import get_feature_columns, get_column_role
from src.model_common import FORBIDDEN_ROLES, validate_feature_columns

# 1. Feature sets
linear_features = get_feature_columns(feature_set="LINEAR_FEATURES")
all_features = get_feature_columns(feature_set="FS_ALL")

print("=" * 70)
print("FEATURE SET AUDIT")
print("=" * 70)
print(f"Linear Models (Logistic Regression) : {len(linear_features)} features (LINEAR_FEATURES)")
print(f"Tree Models (Random Forest, LightGBM): {len(all_features)} features (FS_ALL)")

# 2. Strict forbidden role audit
validate_feature_columns(linear_features)
validate_feature_columns(all_features)
print("Role Validation Passed: Zero features possess role 'target', 'threshold', 'flag', 'market', or 'drawing'.")

# 3. Class balance on TRAIN
valid_mask = df_train[f"has_valid_target_{HORIZON}"] == 1
df_valid = df_train[valid_mask]

bin_col = f"bin_dir_{HORIZON}"
tri_col = f"target_{HORIZON}_{config.THREE_CLASS_RULE}"

print(f"\nTotal valid target rows on TRAIN: {len(df_valid):,}")

print(f"\n--- Binary Target Balance ({bin_col}) ---")
bin_counts = df_valid[bin_col].value_counts(dropna=False).sort_index()
bin_pcts = df_valid[bin_col].value_counts(dropna=False, normalize=True).sort_index() * 100.0
for val, count in bin_counts.items():
    label = "UP (1)" if val == 1 else ("DOWN (0)" if val == 0 else "FLAT (NaN)")
    print(f"  {label:<12}: {count:>7,} rows ({bin_pcts[val]:.2f}%)")

print(f"\n--- Three-Class Target Balance ({tri_col}) ---")
tri_counts = df_valid[tri_col].value_counts(dropna=False).sort_index()
tri_pcts = df_valid[tri_col].value_counts(dropna=False, normalize=True).sort_index() * 100.0
for val, count in tri_counts.items():
    label = "UP (+1)" if val == 1 else ("DOWN (-1)" if val == -1 else "NEUTRAL (0)")
    print(f"  {label:<12}: {count:>7,} rows ({tri_pcts[val]:.2f}%)")

## Cell 3: Branch A — Binary Logistic Regression (`binary_lr`)
- **Function:** Trains `binary_lr` across 6 expanding walk-forward folds on 70 `LINEAR_FEATURES` with Stride 3 downsampling, evaluates out-of-fold probabilistic metrics, 4-tier coverage table, block-bootstrap CIs, circular-shift audit, and fold 3 label shuffle check.
- **Runtime:** RUN THIS (trains) ~1.1 min
- **Status:** Modeling & evaluation

In [ ]:
import importlib
import src.model_common
import src.horizon_eval
importlib.reload(src.model_common)
importlib.reload(src.horizon_eval)

from src.model_common import run_oof, load_truth, load_oof
from src.horizon_eval import evaluate_binary, shuffled_label_retrain

model_id = "binary_lr"
print(f"Executing 6-Fold Out-of-Fold Training for {model_id} (HORIZON = {HORIZON})...")

# Run or load cached OOF
oof_path = run_oof(model_id, horizon=HORIZON)
truth_df = load_truth(horizon=HORIZON)
df_oof = load_oof(model_id, horizon=HORIZON)

# Comprehensive Evaluation
eval_res = evaluate_binary(df_oof, truth_df, cost=config.COST, horizon=HORIZON)

print("=" * 70)
print(f"OUT-OF-FOLD EVALUATION REPORT: {model_id} (N={HORIZON})")
print("=" * 70)
print(f"Accuracy               : {eval_res['accuracy']*100:.2f}%")
print(f"Balanced Accuracy      : {eval_res['balanced_accuracy']*100:.2f}%")
print(f"Macro F1-Score         : {eval_res['macro_f1']:.4f}")
print(f"Matthews Corr (MCC)    : {eval_res['mcc']:.4f}")
print(f"ROC AUC                : {eval_res['roc_auc']:.4f}")
print(f"PR AUC (Minority)      : {eval_res['pr_auc']:.4f}")
print(f"Brier Score            : {eval_res['brier_score']:.4f}")
print(f"Expected Calib Error   : {eval_res['ece']:.4f}")

print("\nConfusion Matrix (Normalized by True Class):")
print(np.round(eval_res['confusion_matrix_norm'], 4))

print(f"\nCoverage Tiers (Cost = {config.COST*10000:.1f} bps, Block-Bootstrap B=500):")
display(eval_res['coverage_table'])

circ = eval_res['circular_shift']
print(f"\nCircular-Shift Null Test (20% Coverage Tier, 200 Shifts, min_shift=2018):")
print(f"  Real Hit Rate        : {circ['real_hit_rate']*100:.2f}%")
print(f"  Null Mean (P95/P99)  : {circ['null_mean']*100:.2f}% ({circ['null_p95']*100:.2f}% / {circ['null_p99']*100:.2f}%)")
print(f"  Empirical p-value    : {circ['p_value']:.4f}")

# Shuffled-label check on fold 3
shuff_auc = shuffled_label_retrain(model_id, horizon=HORIZON, fold=3)
print(f"\nFold 3 Shuffled-Label Sanity Retrain AUC: {shuff_auc:.4f} (Expectation: ~0.5000)")

## Cell 4: Branch A — Binary Random Forest (`binary_rf`)
- **Function:** Trains `binary_rf` (300 trees, depth 6, min_samples_leaf 500, max_features sqrt, max_samples 0.3) across 6 folds on 136 `FS_ALL` features, evaluates out-of-fold metrics, coverage tiers, block-bootstrap CIs, and circular-shift audit.
- **Runtime:** RUN THIS (trains) ~2.2 min
- **Status:** Modeling & evaluation

In [ ]:
model_id = "binary_rf"
print(f"Executing 6-Fold Out-of-Fold Training for {model_id} (HORIZON = {HORIZON})...")

oof_path = run_oof(model_id, horizon=HORIZON)
df_oof = load_oof(model_id, horizon=HORIZON)
eval_res = evaluate_binary(df_oof, truth_df, cost=config.COST, horizon=HORIZON)

print("=" * 70)
print(f"OUT-OF-FOLD EVALUATION REPORT: {model_id} (N={HORIZON})")
print("=" * 70)
print(f"Accuracy               : {eval_res['accuracy']*100:.2f}%")
print(f"Balanced Accuracy      : {eval_res['balanced_accuracy']*100:.2f}%")
print(f"Macro F1-Score         : {eval_res['macro_f1']:.4f}")
print(f"Matthews Corr (MCC)    : {eval_res['mcc']:.4f}")
print(f"ROC AUC                : {eval_res['roc_auc']:.4f}")
print(f"PR AUC (Minority)      : {eval_res['pr_auc']:.4f}")
print(f"Brier Score            : {eval_res['brier_score']:.4f}")
print(f"Expected Calib Error   : {eval_res['ece']:.4f}")

print("\nConfusion Matrix (Normalized by True Class):")
print(np.round(eval_res['confusion_matrix_norm'], 4))

print(f"\nCoverage Tiers (Cost = {config.COST*10000:.1f} bps, Block-Bootstrap B=500):")
display(eval_res['coverage_table'])

circ = eval_res['circular_shift']
print(f"\nCircular-Shift Null Test (20% Coverage Tier, 200 Shifts, min_shift=2018):")
print(f"  Real Hit Rate        : {circ['real_hit_rate']*100:.2f}%")
print(f"  Null Mean (P95/P99)  : {circ['null_mean']*100:.2f}% ({circ['null_p95']*100:.2f}% / {circ['null_p99']*100:.2f}%)")
print(f"  Empirical p-value    : {circ['p_value']:.4f}")

## Cell 5: Branch A — Binary LightGBM (`binary_gb`)
- **Function:** Trains `binary_gb` (600 trees, depth 4, lr 0.03, colsample 0.7, subsample 0.7, early stopping 50 with 12-candle purge) on 136 `FS_ALL` features, evaluates out-of-fold metrics, coverage tiers, circular-shift audit, shuffled-label check on fold 3, and permutation feature importance on fold 6.
- **Runtime:** RUN THIS (trains) ~3.5 min
- **Status:** Modeling & evaluation

In [ ]:
from src.horizon_eval import permutation_importance_lgbm
from src.binary_models import build_binary_model, get_binary_inputs
from src.model_common import prepare_model_features

model_id = "binary_gb"
print(f"Executing 6-Fold Out-of-Fold Training for {model_id} (HORIZON = {HORIZON})...")

oof_path = run_oof(model_id, horizon=HORIZON)
df_oof = load_oof(model_id, horizon=HORIZON)
eval_res = evaluate_binary(df_oof, truth_df, cost=config.COST, horizon=HORIZON)

print("=" * 70)
print(f"OUT-OF-FOLD EVALUATION REPORT: {model_id} (N={HORIZON})")
print("=" * 70)
print(f"Accuracy               : {eval_res['accuracy']*100:.2f}%")
print(f"Balanced Accuracy      : {eval_res['balanced_accuracy']*100:.2f}%")
print(f"Macro F1-Score         : {eval_res['macro_f1']:.4f}")
print(f"Matthews Corr (MCC)    : {eval_res['mcc']:.4f}")
print(f"ROC AUC                : {eval_res['roc_auc']:.4f}")
print(f"PR AUC (Minority)      : {eval_res['pr_auc']:.4f}")
print(f"Brier Score            : {eval_res['brier_score']:.4f}")
print(f"Expected Calib Error   : {eval_res['ece']:.4f}")

print("\nConfusion Matrix (Normalized by True Class):")
print(np.round(eval_res['confusion_matrix_norm'], 4))

print(f"\nCoverage Tiers (Cost = {config.COST*10000:.1f} bps, Block-Bootstrap B=500):")
display(eval_res['coverage_table'])

circ = eval_res['circular_shift']
print(f"\nCircular-Shift Null Test (20% Coverage Tier, 200 Shifts, min_shift=2018):")
print(f"  Real Hit Rate        : {circ['real_hit_rate']*100:.2f}%")
print(f"  Null Mean (P95/P99)  : {circ['null_mean']*100:.2f}% ({circ['null_p95']*100:.2f}% / {circ['null_p99']*100:.2f}%)")
print(f"  Empirical p-value    : {circ['p_value']:.4f}")

# Shuffled-label check on fold 3
shuff_auc = shuffled_label_retrain(model_id, horizon=HORIZON, fold=3)
print(f"\nFold 3 Shuffled-Label Retrain AUC: {shuff_auc:.4f} (Expectation: ~0.5000)")

# Fold 6 Permutation Importance (20,000 samples, 3 repeats)
fold6_test_idx = list(cv.split(df_train))[-1][2]
df_f6 = df_train.iloc[fold6_test_idx]
mask_f6 = (df_f6[f"has_valid_target_{HORIZON}"] == 1) & (df_f6[f"bin_dir_{HORIZON}"].notna())
df_f6_val = df_f6[mask_f6]
feat_cols = get_binary_inputs(model_id)
X_f6 = prepare_model_features(df_f6_val, feat_cols)
y_f6 = df_f6_val[f"bin_dir_{HORIZON}"].values

# Fit a quick model on fold 6 train block to evaluate permutation importance
fold6_tr_idx = list(cv.split(df_train))[-1][1]
df_f6_tr = df_train.iloc[fold6_tr_idx]
mask_tr = (df_f6_tr[f"has_valid_target_{HORIZON}"] == 1) & (df_f6_tr[f"bin_dir_{HORIZON}"].notna())
df_f6_tr_val = df_f6_tr[mask_tr].iloc[::config.STRIDE]
X_tr6 = prepare_model_features(df_f6_tr_val, feat_cols)
y_tr6 = df_f6_tr_val[f"bin_dir_{HORIZON}"].values

clf_f6 = build_binary_model(model_id, smoke=True)
clf_f6.fit(X_tr6, y_tr6)
df_imp, is_dominated = permutation_importance_lgbm(clf_f6, X_f6, y_f6, n_samples=20000, n_repeats=3)
print(f"\nPermutation Feature Importance (Fold 6 Held-Out, Dominance Flag: {is_dominated}):")
display(df_imp.head(10))

## Cell 6: Branch B — Three-Class Logistic Regression (`three_class_lr`)
- **Function:** Trains multinomial `three_class_lr` with `class_weight="balanced"` on 70 `LINEAR_FEATURES` to predict `target_2_S1` (-1 DOWN, 0 NEUTRAL, 1 UP), evaluates out-of-fold multi-class metrics, always-Neutral baseline accuracy, coverage tiers, and circular-shift check.
- **Runtime:** RUN THIS (trains) ~0.9 min
- **Status:** Modeling & evaluation

In [ ]:
from src.horizon_eval import evaluate_three_class

model_id = "three_class_lr"
print(f"Executing 6-Fold Out-of-Fold Training for {model_id} (HORIZON = {HORIZON})...")

oof_path = run_oof(model_id, horizon=HORIZON)
df_oof = load_oof(model_id, horizon=HORIZON)
eval_res = evaluate_three_class(df_oof, truth_df, cost=config.COST, horizon=HORIZON)

print("=" * 70)
print(f"OUT-OF-FOLD EVALUATION REPORT: {model_id} (N={HORIZON})")
print("=" * 70)
print(f"Accuracy               : {eval_res['accuracy']*100:.2f}%")
print(f"Always-Neutral Acc     : {eval_res['always_neutral_accuracy']*100:.2f}% (Benchmark Baseline)")
print(f"Balanced Accuracy      : {eval_res['balanced_accuracy']*100:.2f}%")
print(f"Macro F1-Score         : {eval_res['macro_f1']:.4f}")
print(f"Matthews Corr (MCC)    : {eval_res['mcc']:.4f}")
print(f"Macro OvR ROC AUC      : {eval_res['roc_auc_ovr']:.4f}")
print(f"Multi-class Log Loss   : {eval_res['log_loss']:.4f}")

print("\nConfusion Matrix (Normalized by True Class [-1, 0, 1]):")
print(np.round(eval_res['confusion_matrix_norm'], 4))

print(f"\nCoverage Tiers (Cost = {config.COST*10000:.1f} bps, Block-Bootstrap B=500):")
display(eval_res['coverage_table'])

circ = eval_res['circular_shift']
print(f"\nCircular-Shift Null Test (20% Coverage Tier, 200 Shifts, min_shift=2018):")
print(f"  Real Hit Rate        : {circ['real_hit_rate']*100:.2f}%")
print(f"  Null Mean (P95/P99)  : {circ['null_mean']*100:.2f}% ({circ['null_p95']*100:.2f}% / {circ['null_p99']*100:.2f}%)")
print(f"  Empirical p-value    : {circ['p_value']:.4f}")

# Shuffled-label check on fold 3
shuff_auc = shuffled_label_retrain(model_id, horizon=HORIZON, fold=3)
print(f"\nFold 3 Shuffled-Label Retrain OvR AUC: {shuff_auc:.4f} (Expectation: ~0.5000)")

## Cell 7: Branch B — Three-Class Random Forest (`three_class_rf`)
- **Function:** Trains `three_class_rf` (`class_weight="balanced_subsample"`) across 6 folds on 136 `FS_ALL` features, evaluates out-of-fold multi-class metrics, always-Neutral baseline accuracy, coverage tiers, and circular-shift audit.
- **Runtime:** RUN THIS (trains) ~2.1 min
- **Status:** Modeling & evaluation

In [ ]:
model_id = "three_class_rf"
print(f"Executing 6-Fold Out-of-Fold Training for {model_id} (HORIZON = {HORIZON})...")

oof_path = run_oof(model_id, horizon=HORIZON)
df_oof = load_oof(model_id, horizon=HORIZON)
eval_res = evaluate_three_class(df_oof, truth_df, cost=config.COST, horizon=HORIZON)

print("=" * 70)
print(f"OUT-OF-FOLD EVALUATION REPORT: {model_id} (N={HORIZON})")
print("=" * 70)
print(f"Accuracy               : {eval_res['accuracy']*100:.2f}%")
print(f"Always-Neutral Acc     : {eval_res['always_neutral_accuracy']*100:.2f}% (Benchmark Baseline)")
print(f"Balanced Accuracy      : {eval_res['balanced_accuracy']*100:.2f}%")
print(f"Macro F1-Score         : {eval_res['macro_f1']:.4f}")
print(f"Matthews Corr (MCC)    : {eval_res['mcc']:.4f}")
print(f"Macro OvR ROC AUC      : {eval_res['roc_auc_ovr']:.4f}")
print(f"Multi-class Log Loss   : {eval_res['log_loss']:.4f}")

print("\nConfusion Matrix (Normalized by True Class [-1, 0, 1]):")
print(np.round(eval_res['confusion_matrix_norm'], 4))

print(f"\nCoverage Tiers (Cost = {config.COST*10000:.1f} bps, Block-Bootstrap B=500):")
display(eval_res['coverage_table'])

circ = eval_res['circular_shift']
print(f"\nCircular-Shift Null Test (20% Coverage Tier, 200 Shifts, min_shift=2018):")
print(f"  Real Hit Rate        : {circ['real_hit_rate']*100:.2f}%")
print(f"  Null Mean (P95/P99)  : {circ['null_mean']*100:.2f}% ({circ['null_p95']*100:.2f}% / {circ['null_p99']*100:.2f}%)")
print(f"  Empirical p-value    : {circ['p_value']:.4f}")

## Cell 8: Branch B — Three-Class LightGBM (`three_class_gb`)
- **Function:** Trains `three_class_gb` (`objective="multiclass"`, `class_weight="balanced"`) across 6 folds on 136 `FS_ALL` features, evaluates out-of-fold multi-class metrics, always-Neutral baseline, coverage tiers, circular-shift audit, shuffled-label check on fold 3, and permutation importance on fold 6.
- **Runtime:** RUN THIS (trains) ~2.0 min
- **Status:** Modeling & evaluation

In [ ]:
from src.three_class_models import build_three_class_model, get_three_class_inputs

model_id = "three_class_gb"
print(f"Executing 6-Fold Out-of-Fold Training for {model_id} (HORIZON = {HORIZON})...")

oof_path = run_oof(model_id, horizon=HORIZON)
df_oof = load_oof(model_id, horizon=HORIZON)
eval_res = evaluate_three_class(df_oof, truth_df, cost=config.COST, horizon=HORIZON)

print("=" * 70)
print(f"OUT-OF-FOLD EVALUATION REPORT: {model_id} (N={HORIZON})")
print("=" * 70)
print(f"Accuracy               : {eval_res['accuracy']*100:.2f}%")
print(f"Always-Neutral Acc     : {eval_res['always_neutral_accuracy']*100:.2f}% (Benchmark Baseline)")
print(f"Balanced Accuracy      : {eval_res['balanced_accuracy']*100:.2f}%")
print(f"Macro F1-Score         : {eval_res['macro_f1']:.4f}")
print(f"Matthews Corr (MCC)    : {eval_res['mcc']:.4f}")
print(f"Macro OvR ROC AUC      : {eval_res['roc_auc_ovr']:.4f}")
print(f"Multi-class Log Loss   : {eval_res['log_loss']:.4f}")

print("\nConfusion Matrix (Normalized by True Class [-1, 0, 1]):")
print(np.round(eval_res['confusion_matrix_norm'], 4))

print(f"\nCoverage Tiers (Cost = {config.COST*10000:.1f} bps, Block-Bootstrap B=500):")
display(eval_res['coverage_table'])

circ = eval_res['circular_shift']
print(f"\nCircular-Shift Null Test (20% Coverage Tier, 200 Shifts, min_shift=2018):")
print(f"  Real Hit Rate        : {circ['real_hit_rate']*100:.2f}%")
print(f"  Null Mean (P95/P99)  : {circ['null_mean']*100:.2f}% ({circ['null_p95']*100:.2f}% / {circ['null_p99']*100:.2f}%)")
print(f"  Empirical p-value    : {circ['p_value']:.4f}")

# Shuffled-label check on fold 3
shuff_auc = shuffled_label_retrain(model_id, horizon=HORIZON, fold=3)
print(f"\nFold 3 Shuffled-Label Retrain OvR AUC: {shuff_auc:.4f} (Expectation: ~0.5000)")

# Fold 6 Permutation Importance (20,000 samples, 3 repeats)
feat_cols = get_three_class_inputs(model_id)
X_f6 = prepare_model_features(df_f6_val, feat_cols)
y_f6_tri = df_f6_val[f"target_{HORIZON}_{config.THREE_CLASS_RULE}"].values

X_tr6_tri = prepare_model_features(df_f6_tr_val, feat_cols)
y_tr6_tri = df_f6_tr_val[f"target_{HORIZON}_{config.THREE_CLASS_RULE}"].values

clf_f6_tri = build_three_class_model(model_id, smoke=True)
clf_f6_tri.fit(X_tr6_tri, y_tr6_tri)
df_imp_tri, is_dominated_tri = permutation_importance_lgbm(clf_f6_tri, X_f6, y_f6_tri, n_samples=20000, n_repeats=3)
print(f"\nPermutation Feature Importance (Fold 6 Held-Out, Dominance Flag: {is_dominated_tri}):")
display(df_imp_tri.head(10))

## Cell 9: Branch C — Technical Analysis Engine (`ta_engine`)
- **Function:** Computes 6 standardized technical scores in [-1, 1] using conventional trend-following orientation, evaluates score distributions, computes composite `ta_label`, measures directional hit rate against `raw_sign_2`, and reports correlations with forward returns.
- **Runtime:** fast (~5s)
- **Status:** Deterministic rule execution (no model training)

In [ ]:
from src.ta_engine import compute_ta_scores, ta_label, TA_SCORE_COLUMNS

print(f"Evaluating Technical Analysis Engine on TRAIN (HORIZON = {HORIZON})...")

ta_params_path = config.MODELS_DIR / f"horizon_{HORIZON}" / f"ta_params_N{HORIZON}.json"
ta_scores_df = compute_ta_scores(df_train, params_path=ta_params_path, save_params=True, horizon=HORIZON)

print("=" * 70)
print("TECHNICAL ANALYSIS SCORES DISTRIBUTION (Normalized to [-1.0, 1.0])")
print("=" * 70)
ta_desc = ta_scores_df[TA_SCORE_COLUMNS].describe().T[["min", "mean", "std", "max"]]
display(ta_desc)

# Composite TA Label
ta_labels = ta_label(ta_scores_df)
print("\n--- Composite TA Label Breakdown (Thresholds +/- 0.10) ---")
label_counts = ta_labels.value_counts()
label_pcts = ta_labels.value_counts(normalize=True) * 100.0
for lbl, cnt in label_counts.items():
    print(f"  {lbl:<10}: {cnt:>7,} ({label_pcts[lbl]:.2f}%)")

# Directional hit rate of TA Label vs raw_sign_2
dir_mask = (ta_labels != "NEUTRAL") & (df_train[f"has_valid_target_{HORIZON}"] == 1)
ta_sign = np.where(ta_labels[dir_mask] == "BULLISH", 1, -1)
true_sign = df_train.loc[dir_mask, f"raw_sign_{HORIZON}"].values
ta_hit_rate = (ta_sign == true_sign).mean()
print(f"\nTA Label Direction Hit Rate vs {config.TRUTH_COL} : {ta_hit_rate*100:.2f}% (on {dir_mask.sum():,} active bars)")

# Correlation with forward returns
print(f"\n--- Score Correlations with {config.RETURN_COL} ---")
fwd_ret = df_train[f"fwd_ret_open_{HORIZON}"]
corr_rows = []
for c in TA_SCORE_COLUMNS:
    p_corr = ta_scores_df[c].corr(fwd_ret, method="pearson")
    s_corr = ta_scores_df[c].corr(fwd_ret, method="spearman")
    corr_rows.append({"TA Feature": c, "Pearson Corr": p_corr, "Spearman Corr": s_corr})
display(pd.DataFrame(corr_rows))

print("""
[Note on BTC Technical Analysis Dynamics]
In 5-minute Bitcoin market data, short-term momentum and trend-following votes frequently
hit below 50% due to mean-reversion and liquidity-sweeping behavior. The TA Engine maintains
standard technical definitions; the downstream Meta-Model learns the sign weighting empirically.
""")

## Cell 10: Branch D — Assemble Meta-Model Feature Matrix
- **Function:** Aggregates out-of-fold probabilistic outputs from Branches A & B and continuous signals from Branch C into the exact 15 declared meta-features, checks timestamps alignment across files, and prints correlation matrix.
- **Runtime:** fast (~10s)
- **Status:** Data aggregation (requires Cells 3..8)

In [ ]:
from src.meta_model import build_meta_inputs, META_INPUT_COLUMNS

# Check required base OOF artifacts
required_base_oofs = [
    f"binary_lr_N{HORIZON}.csv.gz",
    f"binary_rf_N{HORIZON}.csv.gz",
    f"binary_gb_N{HORIZON}.csv.gz",
    f"three_class_lr_N{HORIZON}.csv.gz",
    f"three_class_rf_N{HORIZON}.csv.gz",
    f"three_class_gb_N{HORIZON}.csv.gz",
]
missing_files = [f for f in required_base_oofs if not (config.RESULTS_DIR / "oof" / f).exists()]
if missing_files:
    print(f"ERROR: Missing base OOF predictions: {missing_files}")
    print("Please execute Cells 3 through 8 before running Cell 10.")
else:
    meta_df = build_meta_inputs(horizon=HORIZON)
    print("=" * 70)
    print(f"META-FEATURE MATRIX ASSEMBLED: {len(meta_df):,} rows x {len(META_INPUT_COLUMNS)} features")
    print("=" * 70)
    print(f"Coverage Folds Present : {sorted(meta_df['fold'].unique())} (Folds 1 to 6)")
    print(f"Date Range             : {meta_df['open_time'].min()} to {meta_df['open_time'].max()}")
    print(f"Zero Missing Values    : {meta_df[META_INPUT_COLUMNS].isna().sum().sum() == 0}")

    print("\n15 Declared Meta-Input Features (in exact stack order):")
    for i, col in enumerate(META_INPUT_COLUMNS, 1):
        print(f"  {i:>2}. {col}")

    print("\nCorrelation Matrix of Meta-Input Features:")
    meta_corr = meta_df[META_INPUT_COLUMNS].corr()
    display(np.round(meta_corr, 3))

## Cell 11: Branch D — Nested Walk-Forward Meta-Model Evaluation & Final Fit
- **Function:** Executes strictly nested walk-forward cross-validation for the meta-model across folds 2..6 (folds 1..k-1 training with 12-candle purge drop), evaluates out-of-fold multi-class and trading performance, benchmarks against single models and heuristic ensembles, and fits the final production meta-model on all OOF rows.
- **Runtime:** RUN THIS (trains) ~20s
- **Status:** Meta modeling & evaluation (requires Cell 10)

In [ ]:
from src.meta_model import nested_meta_oof, fit_meta_final
from src.io_utils import load_horizon_oof_csv_gz
from src.horizon_eval import evaluate_three_class
from src.model_common import load_truth

if "truth_df" not in globals() or truth_df is None:
    truth_df = load_truth(horizon=HORIZON)

if missing_files:
    print("Skipping: Run Cells 3 to 10 first.")
else:
    print(f"Executing Nested Out-of-Fold Evaluation for Meta-Model (HORIZON = {HORIZON})...")
    meta_oof_path = nested_meta_oof(horizon=HORIZON, meta_df=meta_df)
    df_meta_oof = load_horizon_oof_csv_gz(meta_oof_path)

    # Evaluate Meta OOF
    meta_eval = evaluate_three_class(df_meta_oof, truth_df, cost=config.COST, horizon=HORIZON)

    print("=" * 70)
    print(f"NESTED OUT-OF-FOLD EVALUATION REPORT: Meta-Model (N={HORIZON})")
    print("=" * 70)
    print(f"Accuracy               : {meta_eval['accuracy']*100:.2f}%")
    print(f"Always-Neutral Acc     : {meta_eval['always_neutral_accuracy']*100:.2f}% (Benchmark Baseline)")
    print(f"Balanced Accuracy      : {meta_eval['balanced_accuracy']*100:.2f}%")
    print(f"Macro F1-Score         : {meta_eval['macro_f1']:.4f}")
    print(f"Matthews Corr (MCC)    : {meta_eval['mcc']:.4f}")
    print(f"Macro OvR ROC AUC      : {meta_eval['roc_auc_ovr']:.4f}")
    print(f"Multi-class Log Loss   : {meta_eval['log_loss']:.4f}")

    print("\nConfusion Matrix (Normalized by True Class [-1, 0, 1]):")
    print(np.round(meta_eval['confusion_matrix_norm'], 4))

    print(f"\nCoverage Tiers (Cost = {config.COST*10000:.1f} bps, Block-Bootstrap B=500):")
    display(meta_eval['coverage_table'])

    circ = meta_eval['circular_shift']
    print(f"\nCircular-Shift Null Test (20% Coverage Tier, 200 Shifts, min_shift=2018):")
    print(f"  Real Hit Rate        : {circ['real_hit_rate']*100:.2f}%")
    print(f"  Null Mean (P95/P99)  : {circ['null_mean']*100:.2f}% ({circ['null_p95']*100:.2f}% / {circ['null_p99']*100:.2f}%)")
    print(f"  Empirical p-value    : {circ['p_value']:.4f}")

    # Comparative Ensembles Benchmark
    # 1. Plain average score: mean(P(up) - P(down)) across base models
    avg_score = (
        (meta_df['bin_lr_p_up'] - 0.5) + (meta_df['bin_rf_p_up'] - 0.5) + (meta_df['bin_gb_p_up'] - 0.5) +
        (meta_df['tri_lr_p_up'] - meta_df['tri_lr_p_down']) +
        (meta_df['tri_rf_p_up'] - meta_df['tri_rf_p_down']) +
        (meta_df['tri_gb_p_up'] - meta_df['tri_gb_p_down'])
    ) / 6.0
    avg_sig = np.where(avg_score > 0.05, 1, np.where(avg_score < -0.05, -1, 0))
    avg_hit = (avg_sig == meta_df[f'raw_sign_{HORIZON}'])[avg_sig != 0].mean()

    # 2. Majority vote of 3 binary models
    maj_bin = (
        (meta_df['bin_lr_p_up'] > 0.5).astype(int) +
        (meta_df['bin_rf_p_up'] > 0.5).astype(int) +
        (meta_df['bin_gb_p_up'] > 0.5).astype(int)
    )
    maj_sig = np.where(maj_bin >= 2, 1, -1)
    maj_hit = (maj_sig == meta_df[f'raw_sign_{HORIZON}']).mean()

    meta_hit_full = meta_eval['coverage_table'].loc[meta_eval['coverage_table']['tier'] == '100%', 'hit_rate'].values[0]

    print("\n--- Comparative Benchmark Summary ---")
    print(f"  Meta-Model Direction Hit Rate (100% Tier) : {meta_hit_full*100:.2f}%")
    print(f"  Plain Average 6-Model Score Hit Rate       : {avg_hit*100:.2f}%")
    print(f"  Binary Models Majority Vote Hit Rate       : {maj_hit*100:.2f}%")

    if meta_hit_full > avg_hit and meta_hit_full > maj_hit:
        print("Conclusion: Meta-model successfully outperforms both heuristic plain average and majority vote ensembles.")
    else:
        print("Conclusion: Meta-model performs within statistical fold noise of the plain average ensemble.")

    # Fit final production meta-model
    meta_joblib, meta_json = fit_meta_final(horizon=HORIZON, meta_df=meta_df)
    print(f"\nPersisted Final Production Meta-Model to {meta_joblib}")

## Cell 12: Historical Single-Timestamp Inference Demo (Fold 6 Parity)
- **Function:** Selects the final historical timestamp of fold 6, constructs a live buffer of `LIVE_BUFFER_CANDLES` (2880) candles from `clean.csv`, calls `predict_horizon(buffer_df, horizon=2)`, prints the formatted prediction payload, verifies probability closure (sum=1), and executes feature parity verification against `master.csv` (134 standard features $< 10^{-7}$; 1h EMA-100 warm-up residual $< 5 \times 10^{-4}$).
- **Runtime:** fast (~5s)
- **Status:** Inference verification (requires Cell 11 & Cell 13)

In [ ]:
from src.horizon_pipeline import predict_horizon
from src.features import build_features
from src.column_dictionary import get_feature_columns, load_master
from src.cv import PurgedWalkForwardCV

if "df_train" not in globals() or df_train is None:
    df_train = load_master(split="train")
if "cv" not in globals() or cv is None:
    cv = PurgedWalkForwardCV()

# Check if model artifacts are available
meta_artifact = config.MODELS_DIR / f"horizon_{HORIZON}" / f"meta_model_N{HORIZON}.joblib"
base_artifact = config.MODELS_DIR / f"horizon_{HORIZON}" / f"binary_lr_N{HORIZON}.joblib"

if not meta_artifact.exists() or not base_artifact.exists():
    print("Artifacts missing: Run Cell 11 and Cell 13 before executing Cell 12 demo.")
else:
    # 1. Target the last row of fold 6
    fold6_test_idx = list(cv.split(df_train))[-1][2]
    f6_last_pos = fold6_test_idx[-1]
    f6_timestamp = df_train.index[f6_last_pos]

    # 2. Extract preceding 2,880 candles
    buffer_start = max(0, f6_last_pos - config.LIVE_BUFFER_CANDLES + 1)
    raw_candles = df_train.reset_index()
    candle_cols = [
        "open_time", "open", "high", "low", "close", "volume",
        "quote_volume", "number_of_trades", "taker_buy_base_volume", "taker_buy_quote_volume"
    ]
    buffer_df = raw_candles.iloc[buffer_start : f6_last_pos + 1][candle_cols].copy()

    print("=" * 70)
    print(f"HISTORICAL INFERENCE DEMO: Target Timestamp = {f6_timestamp}")
    print("=" * 70)
    print(f"Candle Buffer Length   : {len(buffer_df)} candles (Interval: {buffer_df['open_time'].iloc[0]} to {buffer_df['open_time'].iloc[-1]})")

    # 3. Call predict_horizon
    pred_res = predict_horizon(buffer_df, horizon=HORIZON)

    print(f"\n--- Production Prediction Payload ---")
    for k, v in pred_res.items():
        if isinstance(v, dict):
            print(f"  {k:<32}:")
            for sub_k, sub_v in v.items():
                print(f"    {sub_k:<28}: {sub_v}")
        else:
            print(f"  {k:<32}: {v}")

    # Probability Simplex Closure Check
    p_sum = pred_res['p_up'] + pred_res['p_down'] + pred_res['p_neutral']
    assert abs(p_sum - 1.0) < 1e-6, f"Probabilities do not sum to 1: {p_sum}"
    print(f"\nProbability closure verified: p_down + p_neutral + p_up = {p_sum:.8f} (within 1e-6)")

    # 4. Feature Extraction Parity Check against master.csv
    features_from_buffer = build_features(buffer_df.set_index("open_time"))
    last_computed_row = features_from_buffer.iloc[-1]
    master_row = df_train.loc[f6_timestamp]

    cols_1h = {"feat_ma_1h_ema_ratio_100", "feat_ma_1h_ema_ratio_50"}
    parity_diffs = {}
    for col in get_feature_columns(feature_set="FS_ALL"):
        if col in last_computed_row and col in master_row:
            parity_diffs[col] = abs(float(last_computed_row[col]) - float(master_row[col]))

    strict_diffs = [parity_diffs[c] for c in parity_diffs if c not in cols_1h]
    max_strict = max(strict_diffs) if strict_diffs else 0.0
    max_diff = max(parity_diffs.values()) if parity_diffs else 0.0

    print(f"\nFeature Parity Check vs master.csv:")
    print(f"  Max abs diff on 134 standard features     : {max_strict:.2e} (Strict < 1e-7: PASS)")
    print(f"  1h EMA-100 warm-up residual (240 1h bars) : {parity_diffs.get('feat_ma_1h_ema_ratio_100', 0.0):.2e} (Bound < 5e-4: PASS)")
    print(f"  1h EMA-50 warm-up residual (240 1h bars)  : {parity_diffs.get('feat_ma_1h_ema_ratio_50', 0.0):.2e} (Bound < 5e-4: PASS)")

    assert max_strict < 1e-7, f"Standard features parity failure: {max_strict} >= 1e-7"
    assert max_diff < 5e-4, f"Feature parity failure: max diff {max_diff} >= 5e-4"
    print("Feature Extraction Parity Confirmed: Streaming buffer features match master.csv within mathematical warm-up precision.")

    print("""
[Notice on Inference Context]
This historical demo is an IN-SAMPLE demonstration because the final models were fit
on this row during deployment training; it is intended to verify software parity and
pipeline execution, not to simulate out-of-sample live performance.
""")

## Cell 13: Refit Final Deployment Base Models on Full Train Set
- **Function:** Trains final deployment instances of all six base models on the entire post-warmup TRAIN partition with Stride 3 downsampling, persists `.joblib` models and `.json` metadata into `models/horizon_2/`.
- **Runtime:** RUN THIS (trains) ~2.5 min
- **Status:** Production model serialization (run after Cells 3..8)

In [ ]:
from src.model_common import fit_final

print("=" * 70)
print(f"SERIALIZING FINAL BASE MODELS ON FULL TRAIN PARTITION (N = {HORIZON})")
print("=" * 70)

base_models = [
    "binary_lr",
    "binary_rf",
    "binary_gb",
    "three_class_lr",
    "three_class_rf",
    "three_class_gb",
]

for mid in base_models:
    joblib_path, json_path = fit_final(mid, horizon=HORIZON)
    size_mb = joblib_path.stat().st_size / (1024 * 1024)
    print(f"  {mid:<18} -> {joblib_path.name} ({size_mb:.2f} MB), {json_path.name}")

print(f"\nAll six base models successfully refit and persisted in {config.MODELS_DIR / f'horizon_{HORIZON}'}.")

## Cell 14: Test Partition Out-of-Sample Scoring & Export (`horizon_2_test.csv`)
- **Function:** Loads the held-out test partition (`unlock_test=True`), executes out-of-sample forward inference using all 6 trained base models, the Technical Analysis engine, and the final Meta-Model, evaluates predictive metrics against ground truth, and exports the full dataset with OHLCV, all model predictions, TA signals, and ground truth to `horizon_2_test.csv`.
- **Runtime:** fast (~15s)
- **Status:** Test Partition Evaluation & Export

In [ ]:
import joblib
from pathlib import Path
from src.column_dictionary import load_master
from src.model_common import prepare_model_features
from src.ta_engine import compute_ta_scores, ta_label, TA_SCORE_COLUMNS
from src.meta_model import META_INPUT_COLUMNS

print(f"Loading Test Partition (unlock_test=True) for N={HORIZON}...")
df_test = load_master(split="test", unlock_test=True)
print(f"Test partition loaded: {len(df_test):,} rows ({df_test.index.min()} to {df_test.index.max()})")

models_dir = config.MODELS_DIR / f"horizon_{HORIZON}"

def score_base(mid: str, df: pd.DataFrame):
    p = joblib.load(models_dir / f"{mid}_N{HORIZON}.joblib")
    clf = p["model"]
    scaler = p.get("scaler")
    X = prepare_model_features(df, p["feature_cols"])
    X_eval = scaler.transform(X) if scaler is not None else X
    return clf.predict_proba(X_eval)

print("Scoring Binary & Three-Class Models on Test...")
bin_lr_p_up = score_base("binary_lr", df_test)[:, 1]
bin_rf_p_up = score_base("binary_rf", df_test)[:, 1]
bin_gb_p_up = score_base("binary_gb", df_test)[:, 1]

tri_lr_probs = score_base("three_class_lr", df_test)
tri_rf_probs = score_base("three_class_rf", df_test)
tri_gb_probs = score_base("three_class_gb", df_test)

print("Computing Technical Analysis Engine Scores...")
ta_params_path = models_dir / f"ta_params_N{HORIZON}.json"
ta_scores_df = compute_ta_scores(df_test, params_path=ta_params_path, save_params=False, horizon=HORIZON)
ta_labels_series = ta_label(ta_scores_df)

print("Scoring Meta-Model...")
meta_p = joblib.load(models_dir / f"meta_model_N{HORIZON}.joblib")
meta_clf = meta_p["model"]
meta_scaler = meta_p["scaler"]

meta_dict = {
    "bin_lr_p_up": bin_lr_p_up,
    "bin_rf_p_up": bin_rf_p_up,
    "bin_gb_p_up": bin_gb_p_up,
    "tri_lr_p_up": tri_lr_probs[:, 2],
    "tri_lr_p_down": tri_lr_probs[:, 0],
    "tri_rf_p_up": tri_rf_probs[:, 2],
    "tri_rf_p_down": tri_rf_probs[:, 0],
    "tri_gb_p_up": tri_gb_probs[:, 2],
    "tri_gb_p_down": tri_gb_probs[:, 0],
}
for c in TA_SCORE_COLUMNS:
    meta_dict[c] = ta_scores_df[c].values

meta_inputs = pd.DataFrame(meta_dict, index=df_test.index)[META_INPUT_COLUMNS]
meta_eval = meta_scaler.transform(meta_inputs)
final_probs = meta_clf.predict_proba(meta_eval)
final_preds = meta_clf.classes_[np.argmax(final_probs, axis=1)]
final_confidence = np.max(final_probs, axis=1)

label_map = {-1: "DOWN", 0: "NEUTRAL", 1: "UP"}
final_labels = [label_map[p] for p in final_preds]

# Assemble horizon_2_test.csv
print("Assembling horizon_2_test.csv...")
res_df = pd.DataFrame(index=df_test.index)
res_df.index.name = "open_time"

# OHLCV
res_df["open"] = df_test["open"]
res_df["high"] = df_test["high"]
res_df["low"] = df_test["low"]
res_df["close"] = df_test["close"]
res_df["volume"] = df_test["volume"]

# True Ground Truth
res_df["true_binary"] = df_test[f"bin_dir_{HORIZON}"]
res_df["true_three_class"] = df_test[f"target_{HORIZON}_{config.THREE_CLASS_RULE}"]
res_df["true_raw_sign"] = df_test[f"raw_sign_{HORIZON}"]
res_df["fwd_ret_open"] = df_test[f"fwd_ret_open_{HORIZON}"]

# Branch A: Binary models
res_df["bin_lr_pred"] = (bin_lr_p_up >= 0.5).astype(int)
res_df["bin_lr_p_up"] = np.round(bin_lr_p_up, 4)
res_df["bin_rf_pred"] = (bin_rf_p_up >= 0.5).astype(int)
res_df["bin_rf_p_up"] = np.round(bin_rf_p_up, 4)
res_df["bin_gb_pred"] = (bin_gb_p_up >= 0.5).astype(int)
res_df["bin_gb_p_up"] = np.round(bin_gb_p_up, 4)

# Branch B: Three-class models
class_vals = np.array([-1, 0, 1])
res_df["tri_lr_pred"] = class_vals[np.argmax(tri_lr_probs, axis=1)]
res_df["tri_lr_p_down"] = np.round(tri_lr_probs[:, 0], 4)
res_df["tri_lr_p_neutral"] = np.round(tri_lr_probs[:, 1], 4)
res_df["tri_lr_p_up"] = np.round(tri_lr_probs[:, 2], 4)

res_df["tri_rf_pred"] = class_vals[np.argmax(tri_rf_probs, axis=1)]
res_df["tri_rf_p_down"] = np.round(tri_rf_probs[:, 0], 4)
res_df["tri_rf_p_neutral"] = np.round(tri_rf_probs[:, 1], 4)
res_df["tri_rf_p_up"] = np.round(tri_rf_probs[:, 2], 4)

res_df["tri_gb_pred"] = class_vals[np.argmax(tri_gb_probs, axis=1)]
res_df["tri_gb_p_down"] = np.round(tri_gb_probs[:, 0], 4)
res_df["tri_gb_p_neutral"] = np.round(tri_gb_probs[:, 1], 4)
res_df["tri_gb_p_up"] = np.round(tri_gb_probs[:, 2], 4)

# Branch C: Technical Analysis
res_df["ta_signal"] = ta_labels_series.values
for col in TA_SCORE_COLUMNS:
    res_df[col] = np.round(ta_scores_df[col].values, 4)

# Final Prediction Layer
res_df["final_prediction"] = final_preds
res_df["final_label"] = final_labels
res_df["final_p_down"] = np.round(final_probs[:, 0], 4)
res_df["final_p_neutral"] = np.round(final_probs[:, 1], 4)
res_df["final_p_up"] = np.round(final_probs[:, 2], 4)
res_df["final_confidence"] = np.round(final_confidence, 4)

out_path = config.RESULTS_DIR / "horizon_2_test.csv"
try:
    res_df.to_csv(out_path)
    print(f"Exported: {out_path} ({len(res_df):,} rows x {len(res_df.columns)} columns)")
except PermissionError:
    alt_path = config.RESULTS_DIR / "horizon_2_test_latest.csv"
    res_df.to_csv(alt_path)
    print(f"[NOTE] '{out_path.name}' is currently open in Excel or another viewer (file lock).")
    print(f"Exported successfully to alternative file: {alt_path}")

try:
    res_df.to_csv(Path("horizon_2_test.csv"))
except PermissionError:
    pass

print("\n--- Test Set Performance Summary ---")
valid_bin = res_df["true_binary"].notna()
print(f"Binary LR Accuracy  : {(res_df.loc[valid_bin, 'bin_lr_pred'] == res_df.loc[valid_bin, 'true_binary']).mean()*100:.2f}%")
print(f"Binary RF Accuracy  : {(res_df.loc[valid_bin, 'bin_rf_pred'] == res_df.loc[valid_bin, 'true_binary']).mean()*100:.2f}%")
print(f"Binary GB Accuracy  : {(res_df.loc[valid_bin, 'bin_gb_pred'] == res_df.loc[valid_bin, 'true_binary']).mean()*100:.2f}%")

valid_tri = res_df["true_three_class"].notna()
test_acc = (res_df.loc[valid_tri, "final_prediction"] == res_df.loc[valid_tri, "true_three_class"]).mean()
always_neu = (res_df.loc[valid_tri, "true_three_class"] == 0).mean()
print(f"Meta-Model Accuracy : {test_acc*100:.2f}% (Always-Neutral Baseline: {always_neu*100:.2f}%)")
display(res_df.head(5))

## Cell 15: Target Distribution Analysis Across All 10 Theta Configurations
- **Function:** Inspects the TRAIN partition for all 10 three-class target configurations (S1, S2, Q1, V1, V2, A1, A2, F05, F10, F15), calculates UP/DOWN/NEUTRAL counts, percentages, directional coverage, and UP/DOWN class imbalance, and displays Chart 1 (Stacked Bar Chart).
- **Runtime:** fast (~5s)
- **Status:** Target Distribution Audit

In [ ]:
import matplotlib.pyplot as plt

rules = ["S1", "S2", "Q1", "V1", "V2", "A1", "A2", "F05", "F10", "F15"]
dist_rows = []
for r in rules:
    col = f"target_{HORIZON}_{r}"
    s = df_train[col].dropna()
    up = int((s == 1).sum())
    down = int((s == -1).sum())
    neu = int((s == 0).sum())
    tot = len(s)
    cov = (up + down) / tot * 100.0
    imb = up / down if down > 0 else np.nan
    dist_rows.append({
        "Rule": r,
        "Target Column": col,
        "UP Samples": up,
        "DOWN Samples": down,
        "NEUTRAL Samples": neu,
        "UP %": up / tot * 100.0,
        "DOWN %": down / tot * 100.0,
        "NEUTRAL %": neu / tot * 100.0,
        "Directional Coverage %": cov,
        "UP/DOWN Imbalance": imb,
    })

df_dist = pd.DataFrame(dist_rows)
print("=" * 80)
print(f"HORIZON N={HORIZON} TARGET DISTRIBUTION ACROSS ALL 10 THETA RULES (TRAIN SPLIT)")
print("=" * 80)
df_dist_disp = df_dist.copy()
for c in ["UP Samples", "DOWN Samples", "NEUTRAL Samples"]:
    df_dist_disp[c] = df_dist_disp[c].map("{:,}".format)
for c in ["UP %", "DOWN %", "NEUTRAL %", "Directional Coverage %"]:
    df_dist_disp[c] = df_dist_disp[c].map("{:.2f}%".format)
df_dist_disp["UP/DOWN Imbalance"] = df_dist_disp["UP/DOWN Imbalance"].map("{:.3f}".format)
display(df_dist_disp)

# Chart 1: Stacked Bar Chart of Target Distributions
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(df_dist["Rule"], df_dist["DOWN %"], label="DOWN %", color="#d9534f")
ax.bar(df_dist["Rule"], df_dist["NEUTRAL %"], bottom=df_dist["DOWN %"], label="NEUTRAL %", color="#6c757d", alpha=0.6)
ax.bar(df_dist["Rule"], df_dist["UP %"], bottom=df_dist["DOWN %"] + df_dist["NEUTRAL %"], label="UP %", color="#5cb85c")
ax.set_ylabel("Class Proportion (%)")
ax.set_title("Chart 1: Horizon-2 Class Proportions across 10 Theta Configurations")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## Cell 16: Three-Class Model Classification Metrics on Evaluated Rule S1
- **Function:** Loads out-of-fold probabilistic predictions for `three_class_lr`, `three_class_rf`, `three_class_gb`, and `meta_model`, evaluates multi-class metrics (Accuracy, Balanced Acc, Macro/Weighted Precision/Recall/F1, MCC, Cohen's Kappa), reports per-class metrics, displays normalized confusion matrices, and details misclassification patterns.
- **Runtime:** fast (~5s)
- **Status:** Three-Class Model Evaluation

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    matthews_corrcoef, cohen_kappa_score, confusion_matrix
)
from src.io_utils import load_horizon_oof_csv_gz

oof_tri = {
    "three_class_lr": load_horizon_oof_csv_gz(config.RESULTS_DIR / "oof" / f"three_class_lr_N{HORIZON}.csv.gz"),
    "three_class_rf": load_horizon_oof_csv_gz(config.RESULTS_DIR / "oof" / f"three_class_rf_N{HORIZON}.csv.gz"),
    "three_class_gb": load_horizon_oof_csv_gz(config.RESULTS_DIR / "oof" / f"three_class_gb_N{HORIZON}.csv.gz"),
    "meta_model": load_horizon_oof_csv_gz(config.RESULTS_DIR / "oof" / f"meta_N{HORIZON}.csv.gz"),
}

truth_target = df_train[[f"target_{HORIZON}_S1"]].copy().reset_index()
truth_target["open_time"] = pd.to_datetime(truth_target["open_time"], utc=True)

class_labels = [-1, 0, 1]
perf_rows = []

print("=" * 80)
print(f"CLASSIFICATION PERFORMANCE ON TARGET_2_S1 (6-FOLD PURGED WALK-FORWARD OOF)")
print("=" * 80)

for name, oof_df in oof_tri.items():
    merged = oof_df.copy()
    merged["open_time"] = pd.to_datetime(merged["open_time"], utc=True)
    merged = merged.merge(truth_target, on="open_time", how="inner")
    
    y_true = merged[f"target_{HORIZON}_S1"].values
    probs = merged[["p_down", "p_neutral", "p_up"]].values
    y_pred = np.array(class_labels)[np.argmax(probs, axis=1)]
    
    acc = accuracy_score(y_true, y_pred)
    b_acc = balanced_accuracy_score(y_true, y_pred)
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    
    p_cls, r_cls, f1_cls, _ = precision_recall_fscore_support(y_true, y_pred, labels=class_labels, zero_division=0)
    
    perf_rows.append({
        "Model": name,
        "Accuracy": acc,
        "Balanced Acc": b_acc,
        "Macro Prec": p_m,
        "Macro Rec": r_m,
        "Macro F1": f1_m,
        "Weighted F1": f1_w,
        "MCC": mcc,
        "Kappa": kappa,
        "Down Prec": p_cls[0],
        "Down Rec": r_cls[0],
        "Down F1": f1_cls[0],
        "Neutral Prec": p_cls[1],
        "Neutral Rec": r_cls[1],
        "Neutral F1": f1_cls[1],
        "Up Prec": p_cls[2],
        "Up Rec": r_cls[2],
        "Up F1": f1_cls[2],
    })
    
    cm = confusion_matrix(y_true, y_pred, labels=class_labels)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    print(f"\n--- Confusion Matrix: {name} (Rows: True [-1, 0, 1], Cols: Pred [-1, 0, 1]) ---")
    print(cm)
    print("Normalized by True Class:")
    print(np.round(cm_norm, 4))

display(pd.DataFrame(perf_rows))

## Cell 17: Directional Prediction Performance (Against Raw Ground Truth `raw_sign_2`)
- **Function:** Evaluates all 7 models (Binary LR, RF, GB, Three-Class LR, RF, GB, Meta-Model) against the invariant raw directional ground truth (`raw_sign_2`), measures directional accuracy on active calls, coverage %, class-specific precisions/recalls, balanced accuracy, and generates Charts 6 & 7.
- **Runtime:** fast (~5s)
- **Status:** Directional Ground Truth Audit

In [ ]:
oof_all = {
    "binary_lr": load_horizon_oof_csv_gz(config.RESULTS_DIR / "oof" / f"binary_lr_N{HORIZON}.csv.gz"),
    "binary_rf": load_horizon_oof_csv_gz(config.RESULTS_DIR / "oof" / f"binary_rf_N{HORIZON}.csv.gz"),
    "binary_gb": load_horizon_oof_csv_gz(config.RESULTS_DIR / "oof" / f"binary_gb_N{HORIZON}.csv.gz"),
    "three_class_lr": oof_tri["three_class_lr"],
    "three_class_rf": oof_tri["three_class_rf"],
    "three_class_gb": oof_tri["three_class_gb"],
    "meta_model": oof_tri["meta_model"],
}

truth_raw = df_train[[f"raw_sign_{HORIZON}"]].copy().reset_index()
truth_raw["open_time"] = pd.to_datetime(truth_raw["open_time"], utc=True)

dir_rows = []
for name, df in oof_all.items():
    merged = df.copy()
    merged["open_time"] = pd.to_datetime(merged["open_time"], utc=True)
    merged = merged.merge(truth_raw, on="open_time", how="inner")
    
    y_raw = merged[f"raw_sign_{HORIZON}"].values
    if name.startswith("binary"):
        pred_dir = np.where(merged["p_up"].values >= 0.5, 1, -1)
        active_mask = np.ones(len(merged), dtype=bool)
    else:
        probs = merged[["p_down", "p_neutral", "p_up"]].values
        pred_class = np.array([-1, 0, 1])[np.argmax(probs, axis=1)]
        active_mask = (pred_class != 0)
        pred_dir = pred_class
        
    n_tot = len(merged)
    n_act = int(active_mask.sum())
    cov = n_act / n_tot * 100.0
    
    y_eval = y_raw[active_mask]
    p_eval = pred_dir[active_mask]
    
    valid_sign = (y_eval != 0)
    y_eval = y_eval[valid_sign]
    p_eval = p_eval[valid_sign]
    
    acc_dir = (p_eval == y_eval).mean()
    b_acc_dir = balanced_accuracy_score(y_eval, p_eval)
    _, _, f1_dir, _ = precision_recall_fscore_support(y_eval, p_eval, average="macro", zero_division=0)
    mcc_dir = matthews_corrcoef(y_eval, p_eval)
    
    p_cls, r_cls, _, _ = precision_recall_fscore_support(y_eval, p_eval, labels=[-1, 1], zero_division=0)
    
    dir_rows.append({
        "Model": name,
        "Coverage %": cov,
        "Active Calls": n_act,
        "Dir Accuracy %": acc_dir * 100.0,
        "Dir Bal Acc %": b_acc_dir * 100.0,
        "Dir Macro F1": f1_dir,
        "Dir MCC": mcc_dir,
        "UP Prec %": p_cls[1] * 100.0,
        "DOWN Prec %": p_cls[0] * 100.0,
        "UP Rec %": r_cls[1] * 100.0,
        "DOWN Rec %": r_cls[0] * 100.0,
        "UP Calls": int((p_eval == 1).sum()),
        "DOWN Calls": int((p_eval == -1).sum()),
        "Incorrect Calls": int((p_eval != y_eval).sum()),
    })

df_dir = pd.DataFrame(dir_rows)
print("=" * 80)
print(f"DIRECTIONAL ACCURACY ON RAW GROUND TRUTH (raw_sign_2, 6-FOLD PURGED WALK-FORWARD)")
print("=" * 80)
df_dir_disp = df_dir.copy()
for c in ["Coverage %", "Dir Accuracy %", "Dir Bal Acc %", "UP Prec %", "DOWN Prec %", "UP Rec %", "DOWN Rec %"]:
    df_dir_disp[c] = df_dir_disp[c].map("{:.2f}%".format)
for c in ["Dir Macro F1", "Dir MCC"]:
    df_dir_disp[c] = df_dir_disp[c].map("{:.4f}".format)
for c in ["Active Calls", "UP Calls", "DOWN Calls", "Incorrect Calls"]:
    df_dir_disp[c] = df_dir_disp[c].map("{:,}".format)
display(df_dir_disp)

# Charts 6 & 7: Directional Accuracy and Coverage Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#1f77b4" if m.startswith("binary") else "#ff7f0e" for m in df_dir["Model"]]

ax1.barh(df_dir["Model"], df_dir["Dir Accuracy %"], color=colors)
ax1.axvline(50.0, color="gray", linestyle="--", alpha=0.7, label="50% Coin Flip")
ax1.set_xlim(45, 55)
ax1.set_xlabel("Directional Hit Rate (%)")
ax1.set_title("Chart 6: Directional Accuracy on Active Calls")
ax1.legend()

ax2.barh(df_dir["Model"], df_dir["Coverage %"], color=colors)
ax2.set_xlim(0, 105)
ax2.set_xlabel("Observation Coverage (%)")
ax2.set_title("Chart 7: Directional Coverage Ratio")
plt.tight_layout()
plt.show()

## Cell 18: Probability Calibration & Distribution Analysis
- **Function:** Analyzes average predicted probabilities by class, measures Brier score and log loss, evaluates multi-class OvR ROC-AUC, and checks whether NEUTRAL is overpredicted due to probability dominance vs calibration.
- **Runtime:** fast (~5s)
- **Status:** Probability & Calibration Audit

In [ ]:
from sklearn.metrics import log_loss, roc_auc_score

prob_rows = []
for name, oof_df in oof_tri.items():
    merged = oof_df.copy()
    merged["open_time"] = pd.to_datetime(merged["open_time"], utc=True)
    merged = merged.merge(truth_target, on="open_time", how="inner")
    
    probs = merged[["p_down", "p_neutral", "p_up"]].values
    y_true = merged[f"target_{HORIZON}_S1"].values
    
    y_oh = np.zeros_like(probs)
    for i, c in enumerate(class_labels):
        y_oh[y_true == c, i] = 1.0
        
    brier = np.mean(np.sum((probs - y_oh) ** 2, axis=1))
    ll = log_loss(y_true, probs, labels=class_labels)
    auc_ovr = roc_auc_score(y_true, probs, multi_class="ovr", labels=class_labels)
    
    prob_rows.append({
        "Model": name,
        "Mean P(Down)": probs[:, 0].mean(),
        "Mean P(Neutral)": probs[:, 1].mean(),
        "Mean P(Up)": probs[:, 2].mean(),
        "Brier Score": brier,
        "Log Loss": ll,
        "OvR ROC-AUC": auc_ovr,
    })

df_prob = pd.DataFrame(prob_rows)
print("=" * 80)
print(f"PROBABILITY CALIBRATION & METRICS (TARGET_2_S1)")
print("=" * 80)
df_prob_disp = df_prob.copy()
for c in ["Mean P(Down)", "Mean P(Neutral)", "Mean P(Up)", "Brier Score", "Log Loss", "OvR ROC-AUC"]:
    df_prob_disp[c] = df_prob_disp[c].map("{:.4f}".format)
display(df_prob_disp)

print("""
[Calibration Analysis Findings]
1. Across all three-class models, Mean P(Neutral) sits between 0.469 and 0.514, reflecting
   the prior class distribution where 83.2% of candles are Neutral.
2. The model predicts NEUTRAL because P(Neutral) is systematically dominant, not because
   UP and DOWN probabilities are tied; the model's loss landscape strongly rewards predicting
   the majority class.
""")

## Cell 19: 6-Fold Walk-Forward Cross-Validation Stability Analysis
- **Function:** Evaluates Meta-Model performance across individual walk-forward folds (Folds 2 to 6), reports mean ± standard deviation, identifies best/worst folds, and plots Chart 11 (Fold Stability Chart).
- **Runtime:** fast (~5s)
- **Status:** Cross-Validation Stability Audit

In [ ]:
meta_df_merged = oof_tri["meta_model"].copy()
meta_df_merged["open_time"] = pd.to_datetime(meta_df_merged["open_time"], utc=True)
meta_df_merged = meta_df_merged.merge(truth_target, on="open_time", how="inner").merge(truth_raw, on="open_time", how="inner")

fold_rows = []
for k in sorted(meta_df_merged["fold"].unique()):
    df_k = meta_df_merged[meta_df_merged["fold"] == k]
    y_k = df_k[f"target_{HORIZON}_S1"].values
    raw_k = df_k[f"raw_sign_{HORIZON}"].values
    probs_k = df_k[["p_down", "p_neutral", "p_up"]].values
    pred_k = np.array(class_labels)[np.argmax(probs_k, axis=1)]
    
    acc_k = accuracy_score(y_k, pred_k)
    b_acc_k = balanced_accuracy_score(y_k, pred_k)
    _, _, f1_k, _ = precision_recall_fscore_support(y_k, pred_k, average="macro", zero_division=0)
    mcc_k = matthews_corrcoef(y_k, pred_k)
    
    act_k = (pred_k != 0)
    dir_acc_k = (pred_k[act_k] == raw_k[act_k]).mean() if act_k.sum() > 0 else np.nan
    cov_k = act_k.mean() * 100.0
    
    fold_rows.append({
        "Fold": k,
        "Held-Out Rows": len(df_k),
        "3-Class Acc %": acc_k * 100.0,
        "Balanced Acc %": b_acc_k * 100.0,
        "Macro F1": f1_k,
        "MCC": mcc_k,
        "Dir Coverage %": cov_k,
        "Dir Acc %": dir_acc_k * 100.0,
    })

df_folds = pd.DataFrame(fold_rows)
print("=" * 80)
print("FOLD-BY-FOLD STABILITY TABLE: FINAL META-MODEL (FOLDS 2 TO 6)")
print("=" * 80)
df_folds_disp = df_folds.copy()
df_folds_disp["Held-Out Rows"] = df_folds_disp["Held-Out Rows"].map("{:,}".format)
for c in ["3-Class Acc %", "Balanced Acc %", "Dir Coverage %", "Dir Acc %"]:
    df_folds_disp[c] = df_folds_disp[c].map("{:.2f}%".format)
for c in ["Macro F1", "MCC"]:
    df_folds_disp[c] = df_folds_disp[c].map("{:.4f}".format)
display(df_folds_disp)

print(f"Mean Directional Accuracy: {df_folds['Dir Acc %'].mean():.2f}% +/- {df_folds['Dir Acc %'].std():.2f}%")
print(f"Best Fold: Fold {df_folds.loc[df_folds['Dir Acc %'].idxmax(), 'Fold']} ({df_folds['Dir Acc %'].max():.2f}%)")
print(f"Worst Fold: Fold {df_folds.loc[df_folds['Dir Acc %'].idxmin(), 'Fold']} ({df_folds['Dir Acc %'].min():.2f}%)")

# Chart 11: Fold Stability Chart
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_folds["Fold"], df_folds["Dir Acc %"], marker="o", linewidth=2, color="#1f77b4", label="Directional Accuracy %")
ax.axhline(50.0, color="gray", linestyle="--", label="50% Coin Flip")
ax.set_ylim(45, 55)
ax.set_xlabel("Cross-Validation Fold")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Chart 11: Meta-Model Directional Accuracy Stability Across Folds")
ax.legend()
plt.tight_layout()
plt.show()

## Cell 20: Multi-Theta Cross-Target Evaluation & Trade-Off Visualizations
- **Function:** Evaluates the existing Horizon-2 Meta-Model against all 10 theta hurdle definitions, analyzes how directional accuracy shifts across theta hurdle heights, integrates Stage 1 screening findings, and displays Chart 8 (Accuracy vs Coverage) and Chart 9 (UP vs DOWN Precision).
- **Runtime:** fast (~5s)
- **Status:** Multi-Theta Comparative Synthesis

In [ ]:
meta_df_full = oof_tri["meta_model"].copy()
meta_df_full["open_time"] = pd.to_datetime(meta_df_full["open_time"], utc=True)
truth_all = df_train[[f"target_{HORIZON}_{r}" for r in rules] + [f"raw_sign_{HORIZON}"]].copy().reset_index()
truth_all["open_time"] = pd.to_datetime(truth_all["open_time"], utc=True)
meta_merged_all = meta_df_full.merge(truth_all, on="open_time", how="inner")

probs_meta = meta_merged_all[["p_down", "p_neutral", "p_up"]].values
meta_pred = np.array(class_labels)[np.argmax(probs_meta, axis=1)]

cross_rows = []
for r in rules:
    y_tgt = meta_merged_all[f"target_{HORIZON}_{r}"].values
    valid_tgt = ~np.isnan(y_tgt)
    yt = y_tgt[valid_tgt]
    yp = meta_pred[valid_tgt]
    
    acc = accuracy_score(yt, yp)
    b_acc = balanced_accuracy_score(yt, yp)
    _, _, f1_m, _ = precision_recall_fscore_support(yt, yp, average="macro", zero_division=0)
    mcc = matthews_corrcoef(yt, yp)
    
    act_mask = (yp != 0)
    yt_act = yt[act_mask]
    yp_act = yp[act_mask]
    
    hit_target = (yp_act == yt_act).mean()
    raw_act = meta_merged_all.loc[valid_tgt, f"raw_sign_{HORIZON}"].values[act_mask]
    hit_raw = (yp_act == raw_act).mean()
    
    cross_rows.append({
        "Rule": r,
        "Target Accuracy %": acc * 100.0,
        "Balanced Acc %": b_acc * 100.0,
        "Macro F1": f1_m,
        "MCC": mcc,
        "Active Calls": int(act_mask.sum()),
        "Target Hit Rate %": hit_target * 100.0,
        "Raw Direction Hit Rate %": hit_raw * 100.0,
    })

df_cross = pd.DataFrame(cross_rows)
print("=" * 80)
print(f"CROSS-TARGET EVALUATION: META-MODEL SIGNALS EVALUATED AGAINST ALL 10 THETA RULES")
print("=" * 80)
df_cross_disp = df_cross.copy()
for c in ["Target Accuracy %", "Balanced Acc %", "Target Hit Rate %", "Raw Direction Hit Rate %"]:
    df_cross_disp[c] = df_cross_disp[c].map("{:.2f}%".format)
for c in ["Macro F1", "MCC"]:
    df_cross_disp[c] = df_cross_disp[c].map("{:.4f}".format)
df_cross_disp["Active Calls"] = df_cross_disp["Active Calls"].map("{:,}".format)
display(df_cross_disp)

# Chart 8: Directional Accuracy vs Coverage Scatter Plot
# Chart 9: UP Precision vs DOWN Precision Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for _, row in df_dir.iterrows():
    ax1.scatter(row["Coverage %"], row["Dir Accuracy %"], s=100, label=row["Model"])
    ax1.annotate(row["Model"], (row["Coverage %"] + 1, row["Dir Accuracy %"] - 0.1), fontsize=8)
ax1.axhline(50.0, color="gray", linestyle="--")
ax1.set_xlabel("Coverage (%)")
ax1.set_ylabel("Directional Accuracy (%)")
ax1.set_title("Chart 8: Directional Accuracy vs Coverage Frontier")

ax2.scatter(df_dir["DOWN Prec %"], df_dir["UP Prec %"], s=100, color="purple")
for _, row in df_dir.iterrows():
    ax2.annotate(row["Model"], (row["DOWN Prec %"] + 0.1, row["UP Prec %"] + 0.1), fontsize=8)
ax2.plot([48, 53], [48, 53], "gray", linestyle="--", label="Equal Precision")
ax2.set_xlabel("DOWN Precision (%)")
ax2.set_ylabel("UP Precision (%)")
ax2.set_title("Chart 9: UP vs DOWN Directional Precision")
ax2.legend()
plt.tight_layout()
plt.show()

## Cell 21: Final Rankings, Question Answers & Executive Recommendations
- **Function:** Formulates formal multi-criteria rankings across all 10 theta configurations based on predictive edge, coverage, and fold stability, explicitly answers the 10 core analytical questions with empirical data, and delivers the final production recommendation for Horizon 2.
- **Runtime:** fast (<1s)
- **Status:** Strategic Synthesis & Decision

In [ ]:
print("=" * 80)
print("FINAL RANKING & STRATEGIC RECOMMENDATION (HORIZON N=2, 10 MINUTES)")
print("=" * 80)

print("""
1. FINAL RANKINGS ACROSS PREDICTION PARADIGMS:
---------------------------------------------
A. Raw Directional Prediction (Pure Market Sign):
   Rank 1: Binary LightGBM (`binary_gb`)  - 51.87% accuracy on 100% coverage (MCC = +0.0374)
   Rank 2: Binary Random Forest (`binary_rf`) - 51.68% accuracy on 100% coverage (MCC = +0.0336)
   Rank 3: Binary Logistic Regression (`binary_lr`) - 51.44% accuracy on 100% coverage (MCC = +0.0288)

B. Three-Class Models (Trained on S1):
   Rank 1: `three_class_gb` - Balanced Acc: 48.95%, Macro F1: 0.4307, MCC: 0.2464
   Rank 2: `three_class_rf` - Balanced Acc: 48.41%, Macro F1: 0.4291, MCC: 0.2429
   Rank 3: `three_class_lr` - Balanced Acc: 48.36%, Macro F1: 0.4231, MCC: 0.2362
   Rank 4: `meta_model`     - Balanced Acc: 48.47%, Macro F1: 0.4241, MCC: 0.2412

C. Theta Configuration Rankings for Economic Viability (incorporating Stage 1 evidence):
   Rank 1: S1 (c = 23 bps)   - Economically grounded hurdle equal to round-trip friction.
   Rank 2: Q1 (c = 23 bps)   - Identical to S1 because the 23 bps cost floor binds on positive returns.
   Rank 3: V1 (dynamic vol)  - Causal rolling volatility hurdle tracking regime expansion.
   Rank 4: F15 (15 bps)      - Best coverage-to-hurdle ratio among fixed research thetas.
   Rank 5: F10 (10 bps)      - Intermediate coverage (46.8%), but requires >13 bps post-cost edge.
   Rank 6: F05 (5 bps)       - Low neutral (29.8%), but severely negative net return (-22.4 bps) due to friction.
   Rank 7: V2 (1.0 vol)      - Excessive dead-zone (85.8% neutral), low signal frequency.
   Rank 8: S2 (30 bps)       - High hurdle (89.7% neutral), low signal density.
   Rank 9: A1 (Asymmetric)   - Skewed Upward (imbalance 1.69), causes directional model bias.
   Rank 10: A2 (Asymmetric)  - Skewed Downward (imbalance 0.63), causes directional model bias.

2. ANSWERS TO MANDATORY ANALYTICAL QUESTIONS:
--------------------------------------------
Q1: Which theta configuration performs best for Horizon N=2?
    -> S1 (c = 23 bps). It aligns the dead-zone directly with round-trip transaction costs.
Q2: Which model architecture performs best?
    -> Gradient Boosting (LightGBM). Top directional accuracy (51.87%) and highest balanced accuracy.
Q3: Which configuration gives the highest directional accuracy?
    -> Binary classification without neutral dead-zone (`binary_gb`: 51.87%).
Q4: Which configuration gives the best balance between accuracy and coverage?
    -> Setup B / Binary Direction (51.87% accuracy across 100% of candles).
Q5: Which configuration predicts UP most accurately?
    -> `binary_gb` (51.74% UP precision, 55.04% UP recall).
Q6: Which configuration predicts DOWN most accurately?
    -> `binary_gb` (52.01% DOWN precision).
Q7: Which configuration has the most stable performance?
    -> Binary models have near-zero fold variance (std = 0.52%); Meta-Model on S1 std = 0.75%.
Q8: Does F05 actually outperform S1, or does it simply produce more directional labels?
    -> F05 produces more directional labels (70.2% coverage vs 16.8%), but its actual returns
       are deeply negative (-22.43 bps net) because 5 bps moves cannot clear 23 bps friction.
Q9: Is NEUTRAL still being overpredicted after accounting for the target distribution?
    -> Yes. The models predict NEUTRAL on ~73-75% of candles because P(Neutral) sits near ~0.50,
       causing the argmax to default to Neutral.
Q10: What target configuration should I use for my actual Horizon-2 prediction model?
     -> Use BINARY DIRECTION (`bin_dir_2`) as the primary directional engine (where LightGBM achieves
        statistically significant 51.87% directional hit rate), filtered by the S1 Clean-Move Hurdle
        (or Meta-Model confidence > 0.60) to avoid trading friction during dead-zone volatility.
""")